In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP3 Fairness-Aware Retraining Candidate v2 notebook
(Tier A, AMPLIFIED reweighting - built after v1's real held-out result, real-run-confirmed
2026-09-23, showed a linear FPR-ratio sample multiplier moved the real FPR ratio only from
0.132381 to 0.139087 - a ~5% relative improvement, nowhere near the real 0.8 four-fifths-style
bar - while costing real PR-AUC (-0.0064) and real recall (-0.0233), and slightly worsening the
real recall-ratio (equal-opportunity) metric (0.95621 -> 0.925479). This v2 candidate squares the
same real per-group FPR ratio before applying it as the real negative-row sample multiplier
(compute_fpr_targeted_sample_weights_v2(), default amplification_exponent=2.0), raising the real
max per-row multiplier from v1's ~7.55x to v2's ~57.06x - now on the same real order of magnitude
as the champion's own real scale_pos_weight (76.58) - a materially stronger real pull toward the
minority Tags groups. Single consolidated code cell (platform convention). Idempotent - safe to
re-run.

Not a numbered gate (Gates 1-6 plus the executive-rollup "Gate 7" are the Master Plan's fixed
governance cycle - PROJECT_STRUCTURE_LOCKED.md does not permit inventing a new gate number). This
is a standalone, additive candidate-evaluation step, run only after the disparate-impact proxy-
feature audit notebook AND the v1 fairness-aware retraining candidate notebook (both already
real-run confirmed), since this notebook reads config blocks both of them wrote. It does NOT
overwrite BP3's real Gate 3-7 artifacts or config blocks, does NOT overwrite v1's own real
config block or artifact, does NOT change the current champion, and does NOT alter any production
decision path - it trains a SEPARATE candidate model, evaluates it on the SAME real held-out test
set, and reports the real three-way trade-off (original champion vs v1 vs v2) for the user's own
governance decision. Nothing here is auto-promoted.

Uses ONLY real data and real, already-confirmed statistics: the real Gold-layer train/test split
(reproducing Gate 3/4/5's identical stratified split), the real shared preprocessing pipeline
(imported from src/features/bp3_escalation_features.py's make_candidates()/
build_shared_preprocessing(), never re-declared inline - HYPER: shared component reuse), and the
real per-group false-positive rate already established by the real disparate-impact investigation
(read live from gate4_disparate_impact_investigation.json, never hardcoded).

Reuses src/models/bp3_fairness_mitigation.py (extended again, this step, with
compute_fpr_targeted_sample_weights_v2()) for every real fairness computation - HYPER: no
fairness-diagnostic or reweighting logic duplicated inline here.
"""

import os, sys, json, time, warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )
    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)
    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402

import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
import yaml  # noqa: E402
from sklearn.metrics import (  # noqa: E402
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split  # noqa: E402
from xgboost import XGBClassifier  # noqa: E402

print = functools.partial(builtins.print, flush=True)

from features.bp3_escalation_features import (  # noqa: E402
    BARRED_COLUMNS,
    COMPANY_COL,
    FEATURE_COLS_CATEGORICAL,
    build_shared_preprocessing,
)
from models.bp3_fairness_mitigation import (  # noqa: E402
    compute_fpr_targeted_sample_weights_v2,
    compute_real_group_confusion_from_predictions,
    FAIRNESS_AWARE_RETRAINING_DISCLOSURE,
)

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp3_complaint_escalation_prediction" / "artifacts"
assert ARTIFACTS_DIR.exists(), f"[CHECK FAILED] {ARTIFACTS_DIR} not found - run BP3 Gates 1-5 first."

# ============================================================
# SECTION 4: Load BP3's real config + the real disparate-impact investigation's per-group FPR +
# v1's own real result (for the three-way comparison) - live, never hardcoded.
# ============================================================
bp3_config_path = CONFIGS_DIR / "bp3_complaint_escalation_prediction.yaml"
with open(bp3_config_path, "r", encoding="utf-8") as f:
    bp3_config = yaml.safe_load(f)

gate3_block = bp3_config.get("gate3_model_benchmark")
proxy_audit_block = bp3_config.get("disparate_impact_proxy_feature_audit")
investigation_block = bp3_config.get("disparate_impact_mitigation_investigation")
v1_candidate_block = bp3_config.get("fairness_aware_retraining_candidate")
assert gate3_block is not None, "[CHECK FAILED] gate3_model_benchmark missing - run BP3 Gate 3 first."
assert investigation_block is not None, (
    "[CHECK FAILED] disparate_impact_mitigation_investigation missing - run the disparate-impact "
    "mitigation investigation notebook first (this candidate extends its MODEL-DRIVEN finding)."
)
assert proxy_audit_block is not None, (
    "[CHECK FAILED] disparate_impact_proxy_feature_audit missing - run the Tier C proxy-feature "
    "audit notebook first (its finding informs this Tier A candidate's design)."
)
assert v1_candidate_block is not None, (
    "[CHECK FAILED] fairness_aware_retraining_candidate (v1) missing - run the v1 fairness-aware "
    "retraining candidate notebook first (v2 is a stronger successor compared directly against it)."
)
assert investigation_block["driver_verdict_category"] == "MODEL-DRIVEN", (
    "[CHECK FAILED] This candidate notebook is written against the real MODEL-DRIVEN diagnosis "
    f"already confirmed twice on real runs; config now records "
    f"'{investigation_block['driver_verdict_category']}' - re-check before proceeding."
)
ORIGINAL_CHAMPION_NAME = gate3_block["champion_model"]
print(f"[OK] Original real champion (Gate 3, real-run-confirmed): {ORIGINAL_CHAMPION_NAME}")
tier_c_tags_meaningful = proxy_audit_block["residual_model_tags_adds_meaningful_information"]
print(
    f"[OK] Tier C real finding (proxy audit): tags_adds_meaningful_information="
    f"{tier_c_tags_meaningful} - no single real feature explains the disparity."
)
print(
    f"[OK] v1 real result (real-run-confirmed): candidate_held_out_test_pr_auc="
    f"{v1_candidate_block['candidate_held_out_test_pr_auc']}, "
    f"candidate_false_positive_rate_ratio_min_over_max="
    f"{v1_candidate_block['candidate_false_positive_rate_ratio_min_over_max']} - v2 attempts a "
    "materially stronger real fairness pull than v1's linear reweighting achieved."
)
RANDOM_STATE = bp3_config["random_state"]
TARGET_COL = bp3_config["target_definition"]["primary_target"]

investigation_json_path = ARTIFACTS_DIR / "gate4_disparate_impact_investigation.json"
assert investigation_json_path.exists(), f"[CHECK FAILED] {investigation_json_path} not found."
with open(investigation_json_path, "r", encoding="utf-8") as f:
    investigation_json = json.load(f)
group_confusion_detail = investigation_json["investigation_summary"]["group_confusion_detail"]
REAL_GROUP_FPR = {row["tags_group"]: row["false_positive_rate"] for row in group_confusion_detail}
REFERENCE_GROUP = "NO_TAG"
assert (
    REFERENCE_GROUP in REAL_GROUP_FPR
), f"[CHECK FAILED] reference group '{REFERENCE_GROUP}' not in real group_confusion_detail."
print(
    f"[OK] Real per-group FPR loaded live from the real investigation "
    f"(reference={REFERENCE_GROUP}): {REAL_GROUP_FPR}"
)

# ============================================================
# SECTION 5: Rebuild the real Gold-layer feature frame + IDENTICAL stratified train/test split as
# Gate 3/4/5 (HYPER reuse of the exact same recipe already proven to reproduce identically) - this
# time also carrying real Tags (never as a model feature - grouping/weighting only).
# ============================================================
GOLD_PATH = DATA_PROCESSED / "cfpb_intervention_escalation_gold.parquet"
assert GOLD_PATH.exists(), f"[CHECK FAILED] {GOLD_PATH} not found - run BP3 Gate 2 first."

select_cols = FEATURE_COLS_CATEGORICAL + [COMPANY_COL, TARGET_COL, "Tags"]
for barred in BARRED_COLUMNS:
    assert barred not in FEATURE_COLS_CATEGORICAL + [
        COMPANY_COL
    ], f"[CHECK FAILED] barred column '{barred}' present in the modeling feature list."
df_pl = pl.scan_parquet(GOLD_PATH).select(select_cols).filter(pl.col(TARGET_COL).is_not_null()).collect()
print(f"[OK] Reloaded real Gold layer, trainable rows: {df_pl.height:,} (must match Gate 3/4/5's row count).")

feature_data = {col: df_pl[col].cast(pl.Utf8).to_list() for col in FEATURE_COLS_CATEGORICAL}
feature_data[COMPANY_COL] = df_pl[COMPANY_COL].cast(pl.Utf8).fill_null("MISSING").to_list()
feature_data["Tags"] = df_pl["Tags"].cast(pl.Utf8).fill_null("NO_TAG").to_list()
target_data = df_pl[TARGET_COL].cast(pl.Int8).to_list()
X_full = pd.DataFrame(feature_data)
y_full = pd.Series(target_data, name=TARGET_COL)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.20, stratify=y_full, random_state=RANDOM_STATE
)
X_train_raw = X_train_raw.reset_index(drop=True)
X_test_raw = X_test_raw.reset_index(drop=True)
y_train = y_train.reset_index(drop=True).to_numpy()
y_test = y_test.reset_index(drop=True).to_numpy()
print(f"[OK] Reproduced Gate 3/4/5's train/test split: train={len(X_train_raw):,}, test={len(X_test_raw):,}.")

n_train_positive = int((y_train == 1).sum())
n_train_negative = int((y_train == 0).sum())
LIVE_SCALE_POS_WEIGHT = n_train_negative / n_train_positive
print(
    f"[OK] Live train-split scale_pos_weight={LIVE_SCALE_POS_WEIGHT:.2f} "
    "(identical formula to Gate 3, recomputed live)."
)

# ============================================================
# SECTION 6: Shared preprocessing (HYPER reuse - identical pipeline to the real champion, fit on
# TRAIN only) + real AMPLIFIED FPR-targeted sample weights (v2's actual fairness intervention).
# ============================================================
tags_train = X_train_raw["Tags"]
tags_test = X_test_raw["Tags"]
X_train_model_cols = X_train_raw[FEATURE_COLS_CATEGORICAL + [COMPANY_COL]]
X_test_model_cols = X_test_raw[FEATURE_COLS_CATEGORICAL + [COMPANY_COL]]

_, X_train_shared, X_test_shared, _, _ = build_shared_preprocessing(X_train_model_cols, X_test_model_cols)
print(f"[OK] Shared preprocessed feature matrix: train={X_train_shared.shape}, test={X_test_shared.shape}.")

AMPLIFICATION_EXPONENT = 2.0
sample_weights = compute_fpr_targeted_sample_weights_v2(
    y_train,
    tags_train,
    REAL_GROUP_FPR,
    reference_group=REFERENCE_GROUP,
    scale_pos_weight=LIVE_SCALE_POS_WEIGHT,
    amplification_exponent=AMPLIFICATION_EXPONENT,
    cap_at_scale_pos_weight=True,
)
print(
    f"[OK] Real AMPLIFIED (exponent={AMPLIFICATION_EXPONENT}) FPR-targeted sample weights computed: "
    f"min={sample_weights.min():.3f}, max={sample_weights.max():.3f}, mean={sample_weights.mean():.3f} "
    f"(v1's real weights were min=1.000, max=76.579, mean=2.147 for comparison)."
)

# ============================================================
# SECTION 7: Fit the v2 fairness-aware candidate - IDENTICAL XGBoost hyperparameters to the real
# champion (Gate 3) and to v1, except the sample_weight array now carries the amplified real
# group-FPR targeting, so this remains an apples-to-apples architectural comparison against both.
# ============================================================
t0 = time.perf_counter()
candidate_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    n_jobs=1,  # exact match to the real champion's own real Gate 3 hyperparameters
    verbosity=0,
    random_state=RANDOM_STATE,
)
candidate_model.fit(X_train_shared, y_train, sample_weight=sample_weights)
fit_seconds = time.perf_counter() - t0
print(f"[OK] v2 amplified fairness-aware candidate fit in {fit_seconds:.1f}s.")

candidate_proba = candidate_model.predict_proba(X_test_shared)[:, 1]
candidate_pred = (candidate_proba >= 0.5).astype(int)

# ============================================================
# SECTION 8: Real evaluation on the SAME real held-out test set - overall metrics (Gate 3's own
# metric set) and real per-group fairness metrics, computed DIRECTLY from real predictions, then a
# real THREE-WAY comparison: original champion vs v1 (real-run-confirmed) vs v2 (this run).
# ============================================================
candidate_metrics = {
    "held_out_test_pr_auc": round(float(average_precision_score(y_test, candidate_proba)), 4),
    "held_out_test_roc_auc": round(float(roc_auc_score(y_test, candidate_proba)), 4),
    "held_out_test_recall": round(float(recall_score(y_test, candidate_pred)), 4),
    "held_out_test_precision": round(float(precision_score(y_test, candidate_pred, zero_division=0)), 4),
    "held_out_test_f1": round(float(f1_score(y_test, candidate_pred, zero_division=0)), 4),
}
print("\n[V2 CANDIDATE] Real held-out test metrics (amplified fairness-aware candidate):")
for k, v in candidate_metrics.items():
    print(f"  {k}={v}")

original_metrics = {
    "held_out_test_pr_auc": gate3_block["held_out_test_pr_auc"],
    "held_out_test_roc_auc": gate3_block["held_out_test_roc_auc"],
    "held_out_test_recall": gate3_block["held_out_test_recall"],
    "held_out_test_precision": gate3_block["held_out_test_precision"],
    "held_out_test_f1": gate3_block["held_out_test_f1"],
}
print("\n[ORIGINAL] Real held-out test metrics (original champion, Gate 3, real-run-confirmed):")
for k, v in original_metrics.items():
    print(f"  {k}={v}")

print("\n[V1] Real held-out test metrics (v1 candidate, real-run-confirmed, from config):")
print(f"  held_out_test_pr_auc={v1_candidate_block['candidate_held_out_test_pr_auc']}")
print(
    f"  candidate_false_positive_rate_ratio_min_over_max="
    f"{v1_candidate_block['candidate_false_positive_rate_ratio_min_over_max']}"
)
print(f"  candidate_recall_ratio_min_over_max={v1_candidate_block['candidate_recall_ratio_min_over_max']}")

group_confusion_candidate = compute_real_group_confusion_from_predictions(tags_test, y_test, candidate_pred)
selection_rates = [row["selection_rate"] for row in group_confusion_candidate]
fprs = [
    row["false_positive_rate"] for row in group_confusion_candidate if row["false_positive_rate"] is not None
]
recalls = [row["recall"] for row in group_confusion_candidate if row["recall"] is not None]
candidate_adverse_impact_ratio = (
    round(min(selection_rates) / max(selection_rates), 4) if max(selection_rates) else None
)
candidate_fpr_ratio = round(min(fprs) / max(fprs), 6) if fprs and max(fprs) else None
candidate_recall_ratio = round(min(recalls) / max(recalls), 6) if recalls and max(recalls) else None

print("\n[V2 CANDIDATE] Real per-group confusion detail (computed directly from real predictions):")
for row in group_confusion_candidate:
    print(
        f"  {row['tags_group']:<32} n={row['n_rows_in_test']:>7,} "
        f"fpr={row['false_positive_rate']} recall={row['recall']} selection_rate={row['selection_rate']}"
    )

original_adverse_impact_ratio = investigation_block["source_adverse_impact_ratio_tags"]
original_fpr_ratio = investigation_block["false_positive_rate_ratio_min_over_max"]
original_recall_ratio = investigation_block["recall_ratio_min_over_max"]
print(
    f"\n[COMPARISON] adverse_impact_ratio_tags (selection-rate ratio): "
    f"original={original_adverse_impact_ratio}, "
    f"v1={v1_candidate_block['candidate_adverse_impact_ratio_tags']}, "
    f"v2={candidate_adverse_impact_ratio}"
)
print(
    f"[COMPARISON] false_positive_rate_ratio_min_over_max: "
    f"original={original_fpr_ratio}, "
    f"v1={v1_candidate_block['candidate_false_positive_rate_ratio_min_over_max']}, "
    f"v2={candidate_fpr_ratio}"
)
print(
    f"[COMPARISON] recall_ratio_min_over_max: "
    f"original={original_recall_ratio}, "
    f"v1={v1_candidate_block['candidate_recall_ratio_min_over_max']}, "
    f"v2={candidate_recall_ratio}"
)
pr_auc_cost_vs_original = round(
    original_metrics["held_out_test_pr_auc"] - candidate_metrics["held_out_test_pr_auc"], 4
)
pr_auc_cost_vs_v1 = round(
    v1_candidate_block["candidate_held_out_test_pr_auc"] - candidate_metrics["held_out_test_pr_auc"], 4
)
print(
    f"[COMPARISON] real PR-AUC cost of v2 vs the original champion: {pr_auc_cost_vs_original} "
    "(positive = v2 is worse)."
)
print(f"[COMPARISON] real PR-AUC cost of v2 vs v1: {pr_auc_cost_vs_v1} (positive = v2 is worse than v1).")

# ============================================================
# SECTION 9: Write the real v2 candidate's artifact + an ADDITIVE config block (never touches Gate
# 3-7's own real, locked blocks, and never touches v1's own real, locked block).
# ============================================================
output_path = ARTIFACTS_DIR / "gate3_fairness_aware_retraining_candidate_v2.json"
output_record = {
    "bp_id": "bp3",
    "investigation": "fairness_aware_retraining_candidate_v2",
    "original_champion_model": ORIGINAL_CHAMPION_NAME,
    "candidate_model": "xgboost_fpr_targeted_reweighted_v2_amplified",
    "amplification_exponent": AMPLIFICATION_EXPONENT,
    "reference_group": REFERENCE_GROUP,
    "real_group_fpr_used_for_weighting": REAL_GROUP_FPR,
    "sample_weight_summary": {
        "min": round(float(sample_weights.min()), 4),
        "max": round(float(sample_weights.max()), 4),
        "mean": round(float(sample_weights.mean()), 4),
    },
    "original_metrics": original_metrics,
    "v1_candidate_metrics": {
        "held_out_test_pr_auc": v1_candidate_block["candidate_held_out_test_pr_auc"],
        "false_positive_rate_ratio_min_over_max": v1_candidate_block[
            "candidate_false_positive_rate_ratio_min_over_max"
        ],
        "recall_ratio_min_over_max": v1_candidate_block["candidate_recall_ratio_min_over_max"],
    },
    "v2_candidate_metrics": candidate_metrics,
    "real_pr_auc_cost_vs_original": pr_auc_cost_vs_original,
    "real_pr_auc_cost_vs_v1": pr_auc_cost_vs_v1,
    "group_confusion_detail_candidate": group_confusion_candidate,
    "original_adverse_impact_ratio_tags": original_adverse_impact_ratio,
    "v1_adverse_impact_ratio_tags": v1_candidate_block["candidate_adverse_impact_ratio_tags"],
    "v2_adverse_impact_ratio_tags": candidate_adverse_impact_ratio,
    "original_false_positive_rate_ratio_min_over_max": original_fpr_ratio,
    "v1_false_positive_rate_ratio_min_over_max": v1_candidate_block[
        "candidate_false_positive_rate_ratio_min_over_max"
    ],
    "v2_false_positive_rate_ratio_min_over_max": candidate_fpr_ratio,
    "original_recall_ratio_min_over_max": original_recall_ratio,
    "v1_recall_ratio_min_over_max": v1_candidate_block["candidate_recall_ratio_min_over_max"],
    "v2_recall_ratio_min_over_max": candidate_recall_ratio,
    "fairness_aware_retraining_disclosure": FAIRNESS_AWARE_RETRAINING_DISCLOSURE,
    "adopted_as_champion": False,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output_record, f, indent=2)
print(f"\n[SAVED] {output_path.relative_to(PROJECT_ROOT)}")

from utils.bp1_config_sync import write_gate_block  # noqa: E402

candidate_marker = (
    "# --- Fairness-Aware Retraining Candidate v2, amplified (Tier A, governance addendum, "
    "appended, idempotent overwrite) ---"
)
CANDIDATE_YAML_BLOCK_LINES = [
    "fairness_aware_retraining_candidate_v2:",
    f'  original_champion_model: "{ORIGINAL_CHAMPION_NAME}"',
    '  candidate_model: "xgboost_fpr_targeted_reweighted_v2_amplified"',
    f"  amplification_exponent: {AMPLIFICATION_EXPONENT}",
    f"  original_held_out_test_pr_auc: {original_metrics['held_out_test_pr_auc']}",
    f"  v1_held_out_test_pr_auc: {v1_candidate_block['candidate_held_out_test_pr_auc']}",
    f"  candidate_held_out_test_pr_auc: {candidate_metrics['held_out_test_pr_auc']}",
    f"  real_pr_auc_cost_vs_original: {pr_auc_cost_vs_original}",
    f"  real_pr_auc_cost_vs_v1: {pr_auc_cost_vs_v1}",
    f"  candidate_adverse_impact_ratio_tags: {candidate_adverse_impact_ratio}",
    f"  candidate_false_positive_rate_ratio_min_over_max: {candidate_fpr_ratio}",
    f"  candidate_recall_ratio_min_over_max: {candidate_recall_ratio}",
    "  adopted_as_champion: false",
    f'  output_artifact: "{output_path.relative_to(PROJECT_ROOT).as_posix()}"',
    f'  generated_at_utc: "{output_record["generated_at_utc"]}"',
]
write_gate_block(bp3_config_path, candidate_marker, CANDIDATE_YAML_BLOCK_LINES)
print(f"[SAVED] {bp3_config_path.relative_to(PROJECT_ROOT)} (fairness_aware_retraining_candidate_v2 block)")

# ============================================================
# SECTION 10: Integrity checks - re-reads the real config fresh off disk (never trusts in-memory
# state) to confirm the write_gate_block() append did not disturb Gate 3's or v1's own real,
# locked blocks.
# ============================================================
with open(bp3_config_path, "r", encoding="utf-8") as f:
    bp3_config_after_write = yaml.safe_load(f)

checks = {
    "original_champion_matches_gate3_recorded": ORIGINAL_CHAMPION_NAME == gate3_block["champion_model"],
    "source_investigation_is_model_driven": investigation_block["driver_verdict_category"] == "MODEL-DRIVEN",
    "no_barred_column_used_as_model_feature": all(
        b not in FEATURE_COLS_CATEGORICAL + [COMPANY_COL] for b in BARRED_COLUMNS
    ),
    "tags_only_used_as_grouping_weighting_variable_not_feature": "Tags"
    not in list(X_train_model_cols.columns),
    "candidate_test_row_count_matches_reconstructed_split": len(y_test) == len(X_test_raw),
    "sample_weights_length_matches_train_rows": len(sample_weights) == len(y_train),
    "sample_weights_all_positive": bool((sample_weights > 0).all()),
    "candidate_metrics_all_present": all(k in candidate_metrics for k in original_metrics),
    "output_artifact_written": output_path.exists(),
    "output_artifact_nonempty": output_path.stat().st_size > 0 if output_path.exists() else False,
    "bp3_config_yaml_updated": "fairness_aware_retraining_candidate_v2" in bp3_config_after_write,
    "original_champion_gate3_block_unchanged": (
        bp3_config_after_write.get("gate3_model_benchmark") == gate3_block
    ),
    "v1_candidate_block_unchanged": (
        bp3_config_after_write.get("fairness_aware_retraining_candidate") == v1_candidate_block
    ),
    "candidate_not_auto_adopted_as_champion": output_record["adopted_as_champion"] is False,
    "disclosure_present_and_nonempty": bool(FAIRNESS_AWARE_RETRAINING_DISCLOSURE),
}

print("\n=== INTEGRITY CHECKS ===")
all_passed = True
for name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"[{status}] {name}")

if all_passed:
    print(
        "\n[ALL CHECKS PASSED] BP3 fairness-aware retraining candidate v2 (amplified) complete. "
        "Fit a real, more strongly FPR-ratio-reweighted XGBoost candidate on the identical real "
        "Gate 3/4/5 train/test split and preprocessing pipeline, evaluated it on the real "
        "held-out test set, and compared it directly against both the real, already-confirmed "
        "original champion AND v1's own real-run-confirmed result. This candidate does NOT "
        "retrain or replace BP3's actual champion, does NOT change Gate 3-7's or v1's "
        "real-run-confirmed outputs, and is NEVER auto-adopted - adopting it is the user's own "
        "real governance decision. Result written to "
        "gate3_fairness_aware_retraining_candidate_v2.json and appended to "
        "bp3_complaint_escalation_prediction.yaml. NEXT STEP: compare the real PR-AUC/recall cost "
        "against the real fairness-ratio improvement, for both v1 and v2, before deciding whether "
        "either candidate should be adopted as BP3's new champion."
    )
else:
    raise AssertionError("[CHECK FAILED] One or more integrity checks failed - see [FAIL] lines above.")
